# Stem cell model for solving network-related problems
Example problem: MTU (maximum transmission unit) on OSI layer 3 was set to 1400 bytes. That prohibits OSPF (a routing protocol) from reaching Full state. Because of that, layer 7 services (RabbitMQ and gRPC) are unable to exchange data. The model receives logs from:
- Cisco IOS (router setup)
- OSPF
- RabbitMQ
- gRPC

and has to transform itself from a generic model to a networking specialist, able to find causal connection between settings set on layer 3 and problems detected at layer 7.

In [1]:
import stem_model
settings = stem_model.Settings()
basic_agent = stem_model.StemModel("gpt-4o", settings.openai_api_key)

## 1. Collecting resources to a single string

In [2]:
context_string = basic_agent.generate_context_string("resources")
print(context_string)

Source cisco-ios-console:
Core-Router-A# show running-config interface GigabitEthernet0/1
Building configuration...

Current configuration : 214 bytes
!
interface GigabitEthernet0/1
 description CONNECTION_TO_DATACENTER_CORE
 mac-address 00ab.cd12.3456
 ip address 10.254.1.1 255.255.255.252
 ip mtu 1400
 ip ospf message-digest-key 1 md5 7 0822455D0A16
 ip ospf dead-interval 40
 ip ospf hello-interval 10
 load-interval 30
 negotiation auto
end

Source gRPC-log:
[2026-05-13T17:48:10.102Z] [ERROR] [BillingSvc] gRPC call to InvoiceProcessor.SubmitBatch failed: rpc error: code = DeadlineExceeded desc = context deadline exceeded
[2026-05-13T17:48:10.105Z] [WARN] [BillingSvc] Payload size (bytes): 658402, attempt: 1/5, initiating retry in 500ms...
[2026-05-13T17:48:10.608Z] [INFO] [BillingSvc] Retrying InvoiceProcessor.SubmitBatch (correlation_id: 8f9a2b1c-4d5e-6f7a)
[2026-05-13T17:48:15.612Z] [ERROR] [BillingSvc] gRPC call to InvoiceProcessor.SubmitBatch failed: rpc error: code = DeadlineExc

## 2. First cell differentiation (first API call)

In [3]:
model_results = basic_agent.differentiate(context_string, 5)
print(model_results)

SPECIALIZATION_NAME:
Network Performance Optimization Agent
----------------------------------------
REASONING_SUMMARY:
The environment telemetry indicates a potential issue with network performance affecting gRPC calls. The gRPC logs show repeated deadline exceeded errors, suggesting network latency or packet loss. The OSPF neighbor state is stuck in EXSTART, which can indicate MTU mismatches or other configuration issues. The MTU on the interface is set to 1400, which might be causing fragmentation issues for large payloads, as seen in the gRPC logs with payloads of 658402 bytes. Additionally, the high message count in the RabbitMQ queue suggests potential network congestion or processing delays.
----------------------------------------
ROOT_CAUSE_HYPOTHESIS:
The root cause of the gRPC deadline exceeded errors is likely due to MTU mismatch or suboptimal MTU settings on the GigabitEthernet0/1 interface, causing packet fragmentation and increased latency. This is supported by the OSPF 

It seems that the model correctly identified the problem and formed a correct hypothesis.

## 3. Create a specialized agent and use it to solve the problem
Here, by solving the problem, I mean selecting appropriate tools and commands for the user to paste (e.g. tool "modify router configuration" and commands from Cisco IOS).

In [4]:
specialized_agent = basic_agent.evolve()

final_results = specialized_agent.generate_remediation(context_string, 5, model_results.success_definition, model_results.root_cause_hypothesis)

print(final_results)

What to do:
1. MODIFY_ROUTER_CONFIGURATION -> 
interface GigabitEthernet0/1
 ip mtu 1500
 end

Rationale:
The current MTU setting on the GigabitEthernet0/1 interface is 1400, which is lower than the standard Ethernet MTU of 1500. This can cause packet fragmentation, leading to increased latency and gRPC deadline exceeded errors. Additionally, the OSPF neighbor is stuck in the EXSTART state, which is often due to MTU mismatches. By increasing the MTU to 1500, we align with the standard Ethernet MTU, reducing the likelihood of fragmentation and improving the chances of the OSPF neighbor transitioning to the FULL state. This should also help in reducing the message backlog in the RabbitMQ queues by improving network performance.



Success! The advanced model correctly identified the issue and provided us a (rationalized!) solution.